# 04 Bioactivity Interpretation

This notebook interprets the AWGC dataset from a materials-informatics perspective.

The goal is not only to predict phase fractions, but also to connect phase evolution to bioactivity-related behavior.

The central idea is:

```text
Processing condition
↓
Initial phase assemblage
↓
SBF-induced phase evolution
↓
Bioactivity-related interpretation
```

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

ROOT_DIR = Path('..')
RAW_DATA_DIR = ROOT_DIR / 'data' / 'raw'
FIGURES_DIR = ROOT_DIR / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

initial_df = pd.read_csv(RAW_DATA_DIR / 'initial_phase_composition.csv')
sbf_df = pd.read_csv(RAW_DATA_DIR / 'sbf_phase_evolution.csv')

initial_df.head(), sbf_df.head()

## Create Bioactivity-Oriented Summary

For this proof-of-concept project, bioactivity is interpreted using phase-evolution trends.

Hydroxyapatite formation is treated as a positive bioactivity indicator.

Wollastonite persistence is treated as an indicator of mechanically relevant crystalline structure and controlled transformation.

The interpretation is qualitative and should be strengthened in future work using pH, degradation, ion-release, and cell-viability datasets.

In [ ]:
day21_df = sbf_df[sbf_df['soaking_day'] == 21].copy()

summary_df = day21_df.merge(
    initial_df,
    on='temperature_C',
    suffixes=('_day21', '_initial')
)

summary_df['hydroxyapatite_gain_pct'] = (
    summary_df['hydroxyapatite_pct_day21'] - summary_df['hydroxyapatite_pct_initial']
)

summary_df['wollastonite_change_pct'] = (
    summary_df['wollastonite_pct_day21'] - summary_df['wollastonite_pct_initial']
)

summary_df[[
    'temperature_C',
    'hydroxyapatite_pct_initial',
    'hydroxyapatite_pct_day21',
    'hydroxyapatite_gain_pct',
    'wollastonite_pct_initial',
    'wollastonite_pct_day21',
    'wollastonite_change_pct'
]]

## Hydroxyapatite Gain After 21 Days

This plot estimates how much the hydroxyapatite fraction changes after 21 days of SBF immersion.

In [ ]:
plt.figure(figsize=(8, 5))

plt.bar(
    summary_df['temperature_C'].astype(str),
    summary_df['hydroxyapatite_gain_pct']
)

plt.xlabel('Sintering temperature (°C)')
plt.ylabel('Hydroxyapatite gain after 21 days (%)')
plt.title('Hydroxyapatite Gain After SBF Immersion')
plt.tight_layout()

plt.savefig(FIGURES_DIR / 'hydroxyapatite_gain_day21.png', dpi=300)
plt.show()

## Simple Bioactivity Labeling

A qualitative bioactivity label is assigned based on experimental interpretation from the Ph.D. work:

- 700 °C: risk / less favorable due to cytotoxicity concern at high extraction concentration
- 800–900 °C: moderate bioactivity
- 1000 °C: high densification and stable response
- 1100 °C: strongest bioactivity response

This labeling is not a universal biological classification. It is a project-specific interpretation for building an AI-readiness demonstration.

In [ ]:
bioactivity_labels = {
    700: 'risk',
    800: 'moderate',
    900: 'moderate',
    1000: 'high_density',
    1100: 'highest_bioactivity'
}

summary_df['bioactivity_interpretation'] = summary_df['temperature_C'].map(bioactivity_labels)

summary_df[[
    'temperature_C',
    'hydroxyapatite_gain_pct',
    'wollastonite_change_pct',
    'bioactivity_interpretation'
]]

## Interpretation

This notebook frames the AWGC dataset as a processing–structure–bioactivity problem.

The key scientific distinction is that 1000 °C and 1100 °C represent different optimization targets:

- 1000 °C is associated with strong densification.
- 1100 °C is associated with the strongest bioactivity response in the experimental study.

This distinction is important because biomaterials design requires balancing mechanical stability, controlled degradation, and bioactivity.

From an AI perspective, the dataset provides a compact proof-of-concept for applying machine learning to experimental materials design.